### Introduction to Data Ingestion

In [104]:
import os
from typing import List,Dict,Any
import pandas as pd

### Import Libraries for Text Splitters

In [105]:
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Setup is done")

Setup is done



### Understanding Document Structure

In [106]:
from langchain_core.documents import Document
doc= Document(
    page_content="This is the main text content that will be embedded and search",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"klinton",
        "date_created":"20-06-2026",
        "custom_field":"any_value"
    }
)
print("Document Structure")

print(f"Content:{doc.page_content}")
print(f"MetaData:{doc.metadata}")


Document Structure
Content:This is the main text content that will be embedded and search
MetaData:{'source': 'example.txt', 'page': 1, 'author': 'klinton', 'date_created': '20-06-2026', 'custom_field': 'any_value'}


### Text Files(.txt)-The Simplest Usecase

In [107]:
import os
os.makedirs("data/text_files",exist_ok=True)

sample_texts={
    "data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language known for its simple, readable syntax that closely resembles 

the English language. 
Created by Guido van Rossum and released in 1991, it emphasizes clean code structure and allows developers 

to express concepts in fewer lines of code than languages like C++ or Java.
Key Features of PythonInterpreted: Python processes code line-by-line, meaning you can run it immediately without a separate compilation step.

Dynamically Typed: You do not need to declare variable types; Python determines them automatically at runtime.Cross-Platform: Python runs seamlessly across different operating systems like Windows, macOS, and Linux.Multi-Paradigm: 

It fully supports procedural, object-oriented (OOP), and functional programming structures.Massive Ecosystem:."""
}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("File content updated successfully")

File content updated successfully


### TextLoader-Read Single File

In [108]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("data/text_files/python_intro.txt",encoding="utf-8")
documents=loader.load()
print(type(documents))
print(documents)

print(f"Loaded {len(documents)} document")
print(f"Content Preview:{documents[0].page_content[:100]}")
print(f"Metadata:{documents[0].metadata}")

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simple, readable syntax that closely resembles \n\nthe English language. \nCreated by Guido van Rossum and released in 1991, it emphasizes clean code structure and allows developers \n\nto express concepts in fewer lines of code than languages like C++ or Java.\nKey Features of PythonInterpreted: Python processes code line-by-line, meaning you can run it immediately without a separate compilation step.\n\nDynamically Typed: You do not need to declare variable types; Python determines them automatically at runtime.Cross-Platform: Python runs seamlessly across different operating systems like Windows, macOS, and Linux.Multi-Paradigm: \n\nIt fully supports procedural, object-oriented (OOP), and functional programming structures.Massive Ecosystem:.')]
Loaded 1 document
Content Preview:Python is a high-level, interpreted progr

### DirectoryLoader - Multiple Text File

In [109]:
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the dirctory
directory_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls= TextLoader, ## Loader class to use
    loader_kwargs= {'encoding':'utf-8'},
    show_progress=True 
)

documents = directory_loader.load()

print(f"Loaded: {len(documents)}  documents")
for i,doc in enumerate(documents):
    print(f"\n Documents {i+1}:")
    print(f"Source:{doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} Characters")

100%|██████████| 2/2 [00:00<00:00, 264.72it/s]

Loaded: 2  documents

 Documents 1:
Source:data\text_files\python_intro.txt
Length: 815 Characters

 Documents 2:
Source:data\text_files\rag_intro.txt
Length: 328 Characters


### Text Spliting Strategies

In [110]:
### Different text splitters techniques

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simple, readable syntax that closely resembles \n\nthe English language. \nCreated by Guido van Rossum and released in 1991, it emphasizes clean code structure and allows developers \n\nto express concepts in fewer lines of code than languages like C++ or Java.\nKey Features of PythonInterpreted: Python processes code line-by-line, meaning you can run it immediately without a separate compilation step.\n\nDynamically Typed: You do not need to declare variable types; Python determines them automatically at runtime.Cross-Platform: Python runs seamlessly across different operating systems like Windows, macOS, and Linux.Multi-Paradigm: \n\nIt fully supports procedural, object-oriented (OOP), and functional programming structures.Massive Ecosystem:.'), Document(metadata={'source': 'data\\text_files\\rag_intro.txt'}, page_content='Retrieval

### Character Text Splitter

In [111]:
# Method 1: CharacterTextSplitter
text=documents[0].page_content
print(f"Original Text Length: {len(text)} Characters")
char_splitter = CharacterTextSplitter(
    separator="\n", ## Split on new lines
    chunk_size=50, ## Max characters per chunk
    chunk_overlap=20, ## Overlap between chunks
    length_function=len ## Function to calculate length (default is len for characters
)
char_chunks = char_splitter.split_text(text)
print(f"Number of Chunks: {len(char_chunks)}")
for i,chunk in enumerate(char_chunks):
    print(f"\nChunk {i+1}:")
    print(f"Chunk Length: {len(chunk)} Characters")
    print(f"Chunk Content:\n{chunk}")

Created a chunk of size 118, which is longer than the specified 50
Created a chunk of size 107, which is longer than the specified 50
Created a chunk of size 75, which is longer than the specified 50
Created a chunk of size 142, which is longer than the specified 50
Created a chunk of size 231, which is longer than the specified 50


Original Text Length: 815 Characters
Number of Chunks: 7

Chunk 1:
Chunk Length: 117 Characters
Chunk Content:
Python is a high-level, interpreted programming language known for its simple, readable syntax that closely resembles

Chunk 2:
Chunk Length: 21 Characters
Chunk Content:
the English language.

Chunk 3:
Chunk Length: 106 Characters
Chunk Content:
Created by Guido van Rossum and released in 1991, it emphasizes clean code structure and allows developers

Chunk 4:
Chunk Length: 75 Characters
Chunk Content:
to express concepts in fewer lines of code than languages like C++ or Java.

Chunk 5:
Chunk Length: 142 Characters
Chunk Content:
Key Features of PythonInterpreted: Python processes code line-by-line, meaning you can run it immediately without a separate compilation step.

Chunk 6:
Chunk Length: 230 Characters
Chunk Content:
Dynamically Typed: You do not need to declare variable types; Python determines them automatically at runtime.Cross-Platform: Python runs seamlessly across

### Recurrsive Text Splitter

In [112]:
# Method 2: RecursiveCharacterTextSplitter
text=documents[0].page_content
print(f"Original Text Length: {len(text)} Characters")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "], ## Try splitting on double newlines, then single newlines, then spaces, and finally characters
    chunk_size=80, ## Max characters per chunk
    chunk_overlap=20, ## Overlap between chunks
    length_function=len ## Function to calculate length (default is len for characters)
)
recursive_chunks = recursive_splitter.split_text(text)
print(f"Number of Chunks: {len(recursive_chunks)}")
for i,chunk in enumerate(recursive_chunks):
    print(f"\nChunk {i+1}:")
    print(f"Chunk Length: {len(chunk)} Characters")
    print(f"Chunk Content:\n{chunk}")

Original Text Length: 815 Characters
Number of Chunks: 14

Chunk 1:
Chunk Length: 78 Characters
Chunk Content:
Python is a high-level, interpreted programming language known for its simple,

Chunk 2:
Chunk Length: 78 Characters
Chunk Content:
for its simple, readable syntax that closely resembles 

the English language.

Chunk 3:
Chunk Length: 71 Characters
Chunk Content:
English language. 
Created by Guido van Rossum and released in 1991, it

Chunk 4:
Chunk Length: 78 Characters
Chunk Content:
in 1991, it emphasizes clean code structure and allows developers 

to express

Chunk 5:
Chunk Length: 69 Characters
Chunk Content:
to express concepts in fewer lines of code than languages like C++ or

Chunk 6:
Chunk Length: 74 Characters
Chunk Content:
like C++ or Java.
Key Features of PythonInterpreted: Python processes code

Chunk 7:
Chunk Length: 73 Characters
Chunk Content:
processes code line-by-line, meaning you can run it immediately without a

Chunk 8:
Chunk Length: 75 Characters
Chunk

### Token Text Splitter

In [113]:
text=documents[0].page_content
token_splitter = TokenTextSplitter(
    chunk_size=50, ## Size in tokens not characters
    chunk_overlap=10 ## Overlap in tokens
)
token_chunks = token_splitter.split_text(text)
print(f"Number of Chunks: {len(token_chunks)}")
for i,chunk in enumerate(token_chunks): 
    print(f"\nChunk {i+1}:")
    print(f"Chunk Length: {len(chunk)} Characters")
    print(f"Chunk Content:\n{chunk}")

Number of Chunks: 5

Chunk 1:
Chunk Length: 250 Characters
Chunk Content:
Python is a high-level, interpreted programming language known for its simple, readable syntax that closely resembles 

the English language. 
Created by Guido van Rossum and released in 1991, it emphasizes clean code structure and allows developers 

Chunk 2:
Chunk Length: 227 Characters
Chunk Content:
, it emphasizes clean code structure and allows developers 

to express concepts in fewer lines of code than languages like C++ or Java.
Key Features of PythonInterpreted: Python processes code line-by-line, meaning you can run

Chunk 3:
Chunk Length: 243 Characters
Chunk Content:
 line-by-line, meaning you can run it immediately without a separate compilation step.

Dynamically Typed: You do not need to declare variable types; Python determines them automatically at runtime.Cross-Platform: Python runs seamlessly across

Chunk 4:
Chunk Length: 232 Characters
Chunk Content:
 runtime.Cross-Platform: Python runs seam